# Numerical Analysis
## Lab 4 — Direct methods for linear systems
*Dr. Marco Fois — A.Y. 2026-2027*

This lab deals with the numerical resolution of a linear system

$$A x = b, \qquad A \in \mathbb{R}^{n \times n}, \quad b \in \mathbb{R}^n .$$

We focus on **direct methods**, i.e. methods for which the solution is obtained in a finite number of operations.

**Contents**

1. Solution of triangular systems
2. LU decomposition
3. Cholesky decomposition
4. Homework
5. Extra homework: exam exercise

### Setup
Run this cell first: it selects a graphics toolkit that can draw inside the notebook (without it, plots fail with the `DISPLAY=''` error).

> **Note.** The functions of this lab are defined in their own cells. Do not use `clear all` inside the notebook: it would also remove the functions defined above. Use `clear` on the variables you want to reset, and `close all` for the figures.

In [1]:
warning("off", "Octave:gnuplot-graphics");
graphics_toolkit("gnuplot");
%plot inline











The FLTK graphics toolkit is not actively maintained and has a number
of limitations that are unlikely to be fixed.
The qt toolkit is recommended instead.











---
# 1. Solution of triangular systems

Consider a generic non singular **lower** triangular system $L x = b$. In order to have $L$ non singular, its diagonal elements must be non zero:

$$\begin{pmatrix}
l_{1,1} & 0 & \cdots & 0\\
l_{2,1} & l_{2,2} & \cdots & 0\\
\vdots & \vdots & \ddots & \vdots\\
l_{n,1} & l_{n,2} & \cdots & l_{n,n}
\end{pmatrix}
\begin{pmatrix} x_1 \\ x_2 \\ \vdots \\ x_n \end{pmatrix}
=
\begin{pmatrix} b_1 \\ b_2 \\ \vdots \\ b_n \end{pmatrix}$$

**Forward substitution**

$$x_1 = \frac{b_1}{l_{11}}, \qquad
x_i = \frac{1}{l_{ii}}\left( b_i - \sum_{j=1}^{i-1} l_{ij} x_j \right), \quad i = 2, \dots, n .$$

Similarly, for a non singular **upper** triangular system $U x = b$:

**Backward substitution**

$$x_n = \frac{b_n}{u_{nn}}, \qquad
x_i = \frac{1}{u_{ii}}\left( b_i - \sum_{j=i+1}^{n} u_{ij} x_j \right), \quad i = n-1, \dots, 1 .$$

Computational cost of both: $O(n^2)$.

In [15]:
function x = forward_substitution(A,b)
  %FORWARD_SUBSTITUTION Solve a lower triangular system using forward
  %substitution method.
  %   x = forward_substitution(A,b)
  %
  %   Inputs  : A = system coefficient matrix
  %             b = right-hand side vector
  %   Outputs : x = solution vector

  n = numel(b);
  x = zeros(size(b));

  for i = 1:n
    numerator = b(i);
    if i > 1
      for j = 1:(i-1)
        numerator = numerator - A(i, j) * x(j);
      end
    end
    x(i) = numerator/A(i, i);
  end
end

In [13]:
function x = backward_substitution(A,b)
  %BACKWARD_SUBSTITUTION Solve an upper triangular system using backward
  %substitution method.
  %   x = backward_substitution(A,b)
  %
  %   Inputs  : A = system coefficient matrix
  %             b = right-hand side vector
  %   Outputs : x = solution vector

  n = numel(b);
  x = zeros(size(b));

  for i = n:-1:1
    numerator = b(i);
    if i < n
      for j = i+1:n
        numerator = numerator - A(i, j) * x(j);
      end
    end
    x(i) = numerator/A(i, i);
  end
end

---
## Exercise 4.1

**a)** Write a function which implements the backward substitution method for solving a generic upper triangular system. Apply such a function to solve the linear system $Ax = b$ with

$$A = \begin{pmatrix} 1&2&3&4\\ 0&1&2&3\\ 0&0&1&2\\ 0&0&0&1 \end{pmatrix}, \qquad b = \begin{pmatrix} 1\\1\\1\\1 \end{pmatrix}.$$

**b)** Write a function to implement the forward substitution method to solve lower triangular systems. Apply this function to the system $A^T x = b$, with $A$ and $b$ defined as above.

**c)** Use the backward and the forward algorithms to solve the system $A^T A x = b$.

In [44]:
%% Exercise 4.1 - backward substitution
close all

n = 4;
A = triu(ones(n) .* (1:n) - (1:n)' + 1)
b = ones(n, 1)

% Check with the builtin \ operator
x_mat = A \ b






A =

   1   2   3   4
   0   1   2   3
   0   0   1   2
   0   0   0   1


b =

   1
   1
   1
   1




x_mat =

   0
   0
  -1
   1



In [45]:
%% Forward substitution: A' is lower triangular
x_fwd = forward_substitution(A', b)

% Check with the builtin \ operator
x_mat2 = A' \ b



x_fwd =

   1
  -1
   0
   0




x_mat2 =

   1
  -1
   0
   0



Since $A^T A x = b$ can be written as $A^T y = b$ followed by $A x = y$, the system is solved with one forward and one backward substitution, without ever forming $A^T A$.

In [46]:
%% Use the backward and the forward algorithms to solve the system A'*A*x = b

y = forward_substitution(A', b)
x = backward_substitution(A, y)

% Check with the builtin \ operator
x_mat3 = (A'*A) \ b




y =

   1
  -1
   0
   0


x =

   3
  -1
   0
   0




x_mat3 =

   3
  -1
   0
   0



---
# 2. LU decomposition

Let $A \in \mathbb{R}^{n \times n}$: the LU decomposition of $A$ allows to write

$$A = LU$$

with $L$ lower triangular such that $l_{ii} = 1$ for all $i = 1, \dots, n$, and $U$ upper triangular.

The **existence and uniqueness** of such a decomposition is related to the principal submatrices of order $i$ of $A$, with $i = 1, \dots, n-1$, which are required to be non singular (necessary and sufficient condition). As an alternative, the following sufficient conditions hold:

- $A$ is strictly diagonally dominant by row, $|a_{ii}| \ge \sum_{j=1, j \neq i}^{n} |a_{ij}|$ for all $i$, or by column;
- $A$ is symmetric positive definite, $A = A^T$ and $x^T A x > 0$ for all $x \neq 0$.

**LU decomposition in MATLAB/Octave**

```
>> help lu
lu   lu factorization
[L,U,P] = lu(A)
returns unit lower triangular matrix L, upper triangular
matrix U, and permutation matrix P so that P*A = L*U.

[L,U] = lu(A)
stores an upper triangular matrix in U and a "psychologically
lower triangular matrix" (i.e. the product of a lower
triangular and a permutation matrix) in L, so that A = L*U.
```

**Without pivoting**

$$\begin{cases} Ax = b \\ A = LU \end{cases} \longrightarrow LUx = b \longrightarrow \begin{cases} Ly = b \\ Ux = y \end{cases}$$

When the standard LU factorization fails, dealing with non-singular matrices, we can resort to the LU factorization **with pivoting**. Pivoting changes the order of the rows of a matrix in order to avoid null pivot elements during the factorization:

$$\begin{cases} Ax = b \\ PA = LU \end{cases} \longrightarrow PAx = Pb \longrightarrow \begin{cases} Ly = Pb \\ Ux = y \end{cases}$$

Notice that the right-hand side of the lower triangular system is modified with respect to the standard LU factorization.

This approach is ideal to solve multiple systems $A x = b_i$ with different right-hand sides sharing the same coefficient matrix: it suffices to compute the decomposition once. Computational cost: $O(n^3)$.

---
## Exercise 4.2

Consider the system $Ax = b$ with

$$A = \begin{pmatrix} 2&10&4&0\\ 1&0&2&2\\ 1&4&0&2\\ 1&2&1&1 \end{pmatrix}, \qquad b = \begin{pmatrix} 10\\1\\3\\3 \end{pmatrix}.$$

**a)** Compute the solution of the system $Ax = b$ using the LU decomposition of matrix $A$.
**b)** Find the determinant of $A$ without using the command `det`.

In [57]:
%% Exercise 4.2
close all

A = [2, 10, 4, 0;
     1, 0, 2, 2;
     1, 4, 0, 2;
     1, 2, 1, 1;
    ];

b = [10; 1; 3; 3];

[L, U, P] = lu(A);

y = forward_substitution(L, P*b);
x = backward_substitution(U, y)

A \ b
















x =

   3.4000
   0.4000
  -0.2000
  -1.0000



ans =

   3.4000
   0.4000
  -0.2000
  -1.0000



In [ ]:
...

% Check with the backslash operator (forward, then backward)
...

% Check by solving the system directly
...

Thanks to the Binet-Cauchy formula, $\det(AB) = \det(A)\det(B)$ for generic square matrices. Since $A = LU$,

$$\det(A) = \det(L)\det(U) = \det(U) = \prod_{i=1}^{n} u_{ii},$$

because $l_{ii} = 1$ for each $i$, so $\det(L) = 1$.

In [58]:
%% Find the determinant of A without using the command det
det_LU = prod(diag(U))

% Check with the command det
det_matlab = det(A)



det_LU = -20



det_matlab = -20


---
## Exercise 4.3

Let us consider the matrices

$$A = \begin{pmatrix} 50&1&3\\ 1&6&0\\ 3&0&1 \end{pmatrix}, \qquad B = \begin{pmatrix} 4&1&1\\ 4&1&2\\ 1&0&15 \end{pmatrix}.$$

1. Apply the command `lu` and compute the LU factorization of the matrices $A$, $B$.
2. Compute the determinant of $A$, $B$ without the command `det` if possible.
3. Solve the systems $Ax = a$, $Bx = b$. Choose the vectors $a$, $b$ such that the solution of the system is $x_{ex} = [1, 1, 1]^T$.
4. Compute the solution by using the backward and forward substitution functions previously implemented.

In [59]:
%% Exercise 4.3
close all

A = [
  50, 1, 3;
  1, 6, 0;
  3, 0, 1;
]

B = [
  4, 1, 1;
  4, 1, 2;
  1, 0, 15;
]









A =

   50    1    3
    1    6    0
    3    0    1







B =

    4    1    1
    4    1    2
    1    0   15



In [64]:
[L_A, U_A, P_A] = lu(A)
prod(diag(U_A))


L_A =

   1.0000        0        0
   0.0200   1.0000        0
   0.0600  -0.0100   1.0000

U_A =

   50.0000    1.0000    3.0000
         0    5.9800   -0.0600
         0         0    0.8194

P_A =

Permutation Matrix

   1   0   0
   0   1   0
   0   0   1


ans = 245


In [66]:
[L_B, U_B, P_B] = lu(B)
det(B)
prod(diag(U_B)/det(P_B))


L_B =

   1.0000        0        0
   0.2500   1.0000        0
   1.0000        0   1.0000

U_B =

    4.0000    1.0000    1.0000
         0   -0.2500   14.7500
         0         0    1.0000

P_B =

Permutation Matrix

   1   0   0
   0   0   1
   0   1   0


ans = 1

ans = 1


The matrix $B$ does not admit a unique LU factorization, because its leading principal submatrix of order 2 is singular:

$$\det \begin{pmatrix} 4&1\\ 4&1 \end{pmatrix} = 0 .$$

This is why `Lb` is not lower triangular — it is the "psychologically lower triangular" matrix of the help text, i.e. a permuted lower triangular factor. Pivoting is required to restore the uniqueness of the factorization.

In [ ]:
...

For the determinant, $P A = L U$ gives $\det(P)\det(A) = \det(U)$, and $\det(P) = \pm 1$ depending on the number of row exchanges, so $\det(A) = \det(U)/\det(P)$.

In [ ]:
% Compute the determinant of A, B without the command det.
% In the case of matrix A we proceed as in Exercise 4.2 (Pa = I)
detA = prod(diag(Ua))
% check
detA_mat = det(A)

% In the case of B, det(P)*det(B) = det(Ub)  --->  det(B) = det(Ub)/det(P)
detB = prod(diag(Ub))/det(Pb)
detB_mat = det(B)

In [ ]:
%% Solve the systems A*x = a, B*x = b with xex = [1,1,1]'
...

In [ ]:
% Now with matrix B: the right-hand side must be permuted, L*y = P*b
...

---
## The inverse matrix

We can define the inverse of a square matrix $A \in \mathbb{R}^{n \times n}$ as the matrix $X = A^{-1}$ such that $AX = XA = I$. It is possible to determine $A^{-1}$ by solving the $n$ linear systems

$$A v_i = e_i, \qquad i = 1, \dots, n,$$

where $e_i$ denote the consecutive columns of the identity matrix (i.e. the vectors of the standard basis of $\mathbb{R}^n$). Thus

$$A^{-1} = [\, v_1 \mid v_2 \mid \dots \mid v_n \,].$$

The LU factorization is computed **once** and reused for the $n$ right-hand sides.

## Exercise 4.4

1. Write a function `[InvA] = MyInv(A)` that computes the inverse of a generic square matrix $A$.
2. Use `MyInv` to compute the inverse of the matrix $A$ of Exercise 4.2. Compare the result with the output of the command `inv`.

In [ ]:
function [InvA] = MyInv(A)

    [L,U] = lu(A);
    %s = size(A);
    %n = s(1);
    n = length(A);

...
...

end

In [ ]:
%% Exercise 4.4
format shortG

A = [2 10 4 0;
     1  0 2 2;
     1 4 0 2;
     1 2 1 1];

InvA = MyInv(A)
InvA_mat = inv(A)

In [ ]:
% How far are the two inverses from each other?
err_inv = norm(InvA - InvA_mat, inf)

format short

---
# 3. Cholesky decomposition

Let $A \in \mathbb{R}^{n \times n}$ be a **symmetric and positive definite** matrix: then there exists a unique $V$ upper triangular, with $v_{ii} > 0$, such that

$$A = V^T V .$$

For any symmetric matrix $A$:

$$A \text{ pos. def.} \iff
\begin{cases}
x^T A x > 0, & \forall x \neq 0\\
\lambda_i > 0, & \forall i = 1, \dots, n, \ \lambda_i \text{ eigenvalue of } A\\
\det(A_i) > 0, & \forall i = 1, \dots, n
\end{cases}$$

$$A \text{ pos. def.} \implies |a_{ij}| < \frac{a_{ii} + a_{jj}}{2}, \quad \forall i,j = 1,\dots,n$$

$$A \text{ pos. def.} \impliedby A \text{ is diagonally dominant.}$$

## Exercise 4.5

Consider the matrix

$$A = \begin{pmatrix}
44 & 15 & 29 & 26 & 119\\
15 & 33 & 32 & 18 & 15\\
29 & 32 & 252 & 112 & 73\\
26 & 18 & 112 & 124 & 90\\
119 & 15 & 73 & 90 & 430
\end{pmatrix}$$

1. Verify that the Cholesky decomposition can be applied to matrix $A$ (the command `eig`);
2. Write a function `[H] = MyChol(A)` to implement the Cholesky decomposition;
3. Compute the Cholesky decomposition by means of both the command `chol` and the function `MyChol`, and compare the results;
4. Solve the linear system $Ax = b$, with $b = [1,1,1,1,1]^T$.

In [ ]:
function [H] = MyChol(A)
    % order of the square matrix A
    n = length(A(1,:));
    H = zeros(n, n);

...
...

end

In [ ]:
%% Exercise 4.5
close all

A = [44 15 29 26 119;
     15 33 32 18 15;
     29 32 252 112 73;
     26 18 112 124 90;
     119 15 73 90 430];
b = [1 1 1 1 1]';

% Verify that the Cholesky decomposition can be applied to matrix A
any(eig(A) <= 0)

The answer is `0`, so no eigenvalue is non-positive: all the eigenvalues are strictly positive and $A$ is positive definite. The symmetry comes from the fact that `eig(A)` returns a real vector, and indeed:

In [ ]:
(A' == A)

In [ ]:
% Mat_R upper triangular
Mat_R = chol(A);
Mat_H = Mat_R'
...
...

Pay attention to the output of the functions: `chol(A)` returns $H^T$ (upper triangular), whereas the function `MyChol` we wrote returns the matrix $H$ (lower triangular).

Now let us solve $H H^T x = b$, with $H$ lower triangular: a forward substitution on $H y = b$, then a backward substitution on $H^T x = y$.

In [ ]:
...

In [ ]:
...

---
# 4. Homework

## Homework 4.1 (LU decomposition)

Consider the system $Ax = b$ with

$$A = \begin{pmatrix} 12&5&-8&-5\\ -4&-4&8&-6\\ 4&2&-3&0\\ 0&-1&2&-4 \end{pmatrix}, \qquad b = \begin{pmatrix} 4\\-6\\3\\-3 \end{pmatrix}.$$

**a)** Use the LU decomposition of matrix $A$ to solve the system. Use both the functions implemented and the `lu` command.
**b)** Compute the determinant of $A$ using the LU decomposition.
**c)** Consider now a different right-hand side vector $c = [-22, -12, -1, -11]^T$. Use an approach which allows us to contain the computational cost.

In [ ]:
% Homework 4.1 - your solution here

## Homework 4.2

Consider the linear system $E x = b$ with

$$E = \begin{pmatrix} 4&1&1&1&5\\ 4&1&2&0&0\\ 1&0&15&5&1\\ 0&2&4&10&2\\ 3&1&2&4&20 \end{pmatrix}, \qquad b = \begin{pmatrix} 12\\19\\22\\18\\30 \end{pmatrix}.$$

1. Verify that the sufficient conditions for the existence and uniqueness of the LU decomposition without pivoting are not fulfilled by matrix $E$.
2. Compute the LU of matrix $E$ with the command `lu` and verify if pivoting takes place with the command `spy`.
3. Solve the system $Ex = b$ by using the LU decomposition of the previous item.

In [ ]:
% Homework 4.2 - your solution here

## Homework 4.3

Consider the system $Ax = b$ with

$$A = \begin{pmatrix} 1&2&1&1\\ 1&4&0&2\\ 2&10&4&0\\ 1&0&2&2 \end{pmatrix}, \qquad b = \begin{pmatrix} 3\\3\\10\\1 \end{pmatrix}.$$

**a)** Compute the solution of the system using the LU decomposition **with pivoting** of matrix $A$.
**b)** Find the determinant of $A$ (use the command `det` only to compute the determinant of $P$).

In [ ]:
% Homework 4.3 - your solution here

## Homework 4.4

Consider the system $Ax = b$ with

$$A = \begin{pmatrix} 10&0&3&0\\ 0&5&0&-2\\ 3&0&5&0\\ 0&-2&0&2 \end{pmatrix}, \qquad b = \begin{pmatrix} 2\\2\\2\\2 \end{pmatrix}.$$

**a)** Compute the solution of the system. Use the Cholesky decomposition of matrix $A$ if possible (check the hypotheses).
**b)** Find the determinant of $A$ without using the command `det`.

In [ ]:
% Homework 4.4 - your solution here

## Homework 4.5

Consider the tridiagonal matrix $A_n \in \mathbb{R}^{n \times n}$ defined as

$$A_n = \begin{pmatrix}
a & b & & &\\
b & a & b & &\\
& \ddots & \ddots & \ddots &\\
& & b & a & b\\
& & & b & a
\end{pmatrix}$$

for $a = 2$ and $b = 1$.

**a)** Verify that $A_{10}$ is a symmetric positive definite matrix.
**b)** Provide the form of the matrix $V_{10}$ such that $A_{10} = V_{10}^T V_{10}$.

In [ ]:
% Homework 4.5 - your solution here

## Homework 4.6

Consider the linear system $Ax = b$ with

$$A = \begin{pmatrix}
1 & 10^{10} & 1 & 1\\
10^{10} & 1 & 1 & 10^{10}\\
1 & 1 & 10^{-10} & 1\\
1 & 10^{10} & 1 & 10^{10}
\end{pmatrix}, \qquad
b = \begin{pmatrix} 10^{10} + 3\\ 2\cdot 10^{10} + 2\\ 3 + 10^{-10}\\ 2 \cdot 10^{10} + 2 \end{pmatrix}$$

such that the exact solution is $x = [1,1,1,1]^T$.

**a)** Solve the system using LU decomposition with pivoting.
**b)** Compute the infinity norm of the error associated with the solution of the previous item.
**c)** Can we apply the Cholesky decomposition to matrix $A$?
**d)** Is the normalized residual $\|b - Ax\| / \|b\|$ a good estimator for the relative error? Motivate your answer.

In [ ]:
% Homework 4.6 - your solution here

---
# 5. Extra homework: exam exercise

Consider the following family of tri-diagonal matrices $A_n(a) \in \mathbb{R}^{n \times n}$

$$A_n(a) = \begin{pmatrix}
1 & -6 & 0 & 0 & \cdots\\
-6 & 1 & -7 & 0 & \cdots\\
0 & -7 & 1 & -8 & \cdots\\
0 & 0 & -8 & \ddots & \ddots\\
\vdots & \vdots & \vdots & \ddots & \ddots
\end{pmatrix} + a\, I_n$$

$I_n$ denoting the identity matrix of order $n$.

**a)** Use the built-in commands `diag` and `eye` to assemble the matrix $A_7(32)$.
**b)** Using the built-in command `lu`, compute the LU factorization of $A_7(32)$ and then, using `nnz` and `spy`, provide a motivated answer to the following questions: was pivoting performed during the factorization? Did fill-in occur?
**c)** Let $x_{ex} = [1, 1, 1, \dots]^T \in \mathbb{R}^n$ and $b_n(a) = A_n(a) x_{ex}$. Use the function `jacobi.m` to solve the system $A_7(1)x = b_7(1)$ by Jacobi's iterative method and compute the error $\eta_7(1) = \|x_{ex} - x\|_2$; set `tol = 1e-6` and `nmax = 100`, with the zero vector as initial guess. Comment the results based on theoretical expectations.
**d)** Same as c) for the system $A_7(32) x = b_7(32)$, and compare. *(N.B.: $b_7(32) \neq b_7(1)$, so the right-hand side must be recomputed.)*

> Points c) and d) use the function `jacobi.m` of the next lab, which is not included in this notebook.

### a) Assembling the matrix

In [ ]:
%% Extra homework - point a)
close all

% In a notebook, set the values directly instead of using input()
n = 7;    % n = input('Enter the size n of the matrix: ');
a = 32;   % a = input('Enter the parameter a: ');

% --- Diagonals definition ---
main_diag = ones(n,1);               % main diagonal entries = 1
start = -6;                          % decreasing from -6
len_diag = n-1;                      % length of sub/super diag

sub_diag   = start : -1 : start - (len_diag - 1);   % sub diagonal
super_diag = sub_diag;                              % super diagonal

% --- Assemble the tri-diagonal matrix ---
A = diag(main_diag,0) + ...
    diag(sub_diag,-1) + ...
    diag(super_diag,1);

% --- Add a*I_n ---
A = A + a*eye(n);

% --- Display result ---
disp('Matrix A_n(a) = ')
disp(A)

### b) LU factorization, pivoting and fill-in

In [ ]:
%% point b)
% ----- LU factorization -----
[L,U,P] = lu(A);

% ----- Check pivoting -----
if isequal(P, eye(n))
    disp('No pivoting was performed (P = I).')
else
    disp('Pivoting was performed (P is different from I).')
end

In [ ]:
% ----- Check fill-in -----
nnzA = nnz(A);     % number of non-zero entries in A
nnzL = nnz(L);
nnzU = nnz(U);

fprintf('Non-zero entries:\n A = %d\n L = %d\n U = %d\n', nnzA, nnzL, nnzU);

% L and U share the diagonal positions of A, and the unit diagonal of L is
% counted in nnzL: the pattern of L+U covers nnzL + nnzU - n positions.
if (nnzL + nnzU - n) > nnzA
    disp('Fill-in occurred (L and U have non-zeros where A had zeros).')
else
    disp('No fill-in occurred.')
end

In [ ]:
% ----- Visualize sparsity -----
figure;
subplot(1,3,1); spy(A); title('A_7(32)');
subplot(1,3,2); spy(L); title('L factor');
subplot(1,3,3); spy(U); title('U factor');

**Comment.** The matrix is strictly diagonally dominant by rows: the diagonal entries are $1 + 32 = 33$, while the off-diagonal entries of each row sum at most to $|-10| + |-11| = 21 < 33$. Diagonal dominance guarantees the existence of the LU factorization without pivoting, and indeed `P` is the identity.

About fill-in: $A$ is tridiagonal with $\text{nnz}(A) = 19$, and the factors are bidiagonal, $\text{nnz}(L) = \text{nnz}(U) = 13$. Counting the shared diagonal only once, $13 + 13 - 7 = 19 = \text{nnz}(A)$: the factors keep the band structure of $A$ and **no fill-in occurs**. The three `spy` plots show the same pattern, split into its lower and upper parts.

In [1]:
% Salva prima il notebook (Ctrl+S)
system('..\.venv\Scripts\jupyter nbconvert --to pdf "Lab4_Direct_Methods_Lecture.ipynb"');

[NbConvertApp] Converting notebook Lab4_Direct_Methods_Lecture.ipynb to pdf
[NbConvertApp] Writing 62059 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | b had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 99861 bytes to Lab4_Direct_Methods_Lecture.pdf
